In [ ]:
# ================================================
# COMPLETE: Fast Unzip + Landmark Extraction
# Copy zip to local /content/ for SSD speed
# ================================================

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install packages
!pip install mediapipe tqdm opencv-python-headless -q

# 3. Imports
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import csv
import os
from tqdm import tqdm

# =================== PATHS (updated for local copy) ===================
drive_zip_path   = "/content/drive/MyDrive/Colab_Notebooks/EDGE/archive.zip"          # original on Drive
local_zip_path   = "/content/archive.zip"                                             # copy here
local_extract_path = "/content/archive"                                               # unzip here (fast SSD)
dataset_root     = os.path.join(local_extract_path, "asl_alphabet_train", "asl_alphabet_train")  # inner folder with A/B/.../Z
output_csv       = os.path.join(local_extract_path, "asl_landmarks.csv")              # CSV on fast disk
model_path       = "/content/drive/MyDrive/Colab_Notebooks/EDGE/hand_landmarker.task" # model can stay on Drive
# ======================================================================

# 4. Download model (only once, keep on Drive)
if not os.path.exists(model_path):
    print("Downloading Hand Landmarker model...")
    !wget -q -O "{model_path}" "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    print("✅ Model downloaded!\n")

# 5. Copy zip from Drive to local disk (speeds up unzip dramatically)
print("Copying zip from Drive to local disk...")
!cp "{drive_zip_path}" "{local_zip_path}"
print("✅ Copy complete!\n")

# 6. Unzip on local disk (much faster than unzipping directly from Drive)
print("Unzipping on local SSD...")
!unzip -q "{local_zip_path}" -d "{local_extract_path}"
print("✅ Unzip complete!\n")

# Optional cleanup: remove local zip to save space (after unzip succeeds)
# !rm "{local_zip_path}"

# 7. Setup MediaPipe Tasks API
BaseOptions = python.BaseOptions
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

landmarker = HandLandmarker.create_from_options(options)

# 8. Create output folder if needed
os.makedirs(local_extract_path, exist_ok=True)

# =================== DEBUG PATH CHECK ===================
print("\n🔍 PATH CHECK")
print(f"Looking for A-Z folders in: {dataset_root}")
if os.path.exists(dataset_root):
    folders = [f for f in os.listdir(dataset_root) if os.path.isdir(os.path.join(dataset_root, f))]
    print(f"Found {len(folders)} folders")
    print("Sample folders (first 10):", folders[:10])
else:
    print("❌ dataset_root not found! Listing local_extract_path contents:")
    !ls -la "{local_extract_path}"

# =================== EXTRACTION ===================
with open(output_csv, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    header = [f"{coord}{i}" for i in range(21) for coord in ['x', 'y', 'z']] + ['label']
    writer.writerow(header)

    total_detected = 0
    print("\nStarting landmark extraction...")

    for letter in tqdm("ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
        # Skip non A-Z (though loop is already A-Z)
        if letter not in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
            continue

        folder = os.path.join(dataset_root, letter)
        if not os.path.exists(folder):
            continue

        img_count = 0
        detected_in_letter = 0
        for img_file in os.listdir(folder):
            if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            img_count += 1

            img_path = os.path.join(folder, img_file)
            image = cv2.imread(img_path)
            if image is None:
                continue

            rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

            detection_result = landmarker.detect(mp_image)

            if detection_result.hand_landmarks:
                lm = detection_result.hand_landmarks[0]
                row = []
                for landmark in lm:
                    row.extend([landmark.x, landmark.y, landmark.z])
                row.append(letter)
                writer.writerow(row)
                total_detected += 1
                detected_in_letter += 1

        if img_count > 0:
            print(f"  {letter}: {img_count} images → {detected_in_letter} hands detected")

print(f"\n✅ Extraction finished!")
print(f"CSV saved to: {output_csv}")
print(f"Total rows extracted: {total_detected}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 17.7 MB/s eta 0:00:00
Copying zip from Drive to local disk...
✅ Copy complete!

Unzipping on local SSD...
✅ Unzip complete!


🔍 PATH CHECK
Looking for A-Z folders in: /content/archive/asl_alphabet_train/asl_alphabet_train
Found 29 folders
Sample folders (first 10): ['T', 'del', 'N', 'H', 'Q', 'A', 'P', 'M', 'W', 'U']

Starting landmark extraction...


  4%|▍         | 1/26 [01:22<34:16, 82.27s/it]

  A: 3000 images → 2143 hands detected


  8%|▊         | 2/26 [02:44<32:54, 82.28s/it]

  B: 3000 images → 2172 hands detected


...

  X: 3000 images → 2122 hands detected


 96%|█████████▌| 25/26 [34:47<01:24, 84.95s/it]

  Y: 3000 images → 2559 hands detected


100%|██████████| 26/26 [36:11<00:00, 83.52s/it]

  Z: 3000 images → 2307 hands detected

✅ Extraction finished!
CSV saved to: /content/archive/asl_landmarks.csv
Total rows extracted: 58997


In [ ]:
!cp "{output_csv}" "/content/drive/MyDrive/Colab_Notebooks/EDGE/"

In [ ]:
# =================== EXTRACTION FOR SPECIAL CLASSES ONLY ===================
special_classes = ["space", "del"]   # or just ["SPACE"] if you only want spacing

with open("/content/archive/special_landmarks.csv", 'w', newline='') as csvfile:  # new CSV
    writer = csv.writer(csvfile)
    header = [f"{coord}{i}" for i in range(21) for coord in ['x', 'y', 'z']] + ['label']
    writer.writerow(header)

    total_detected_special = 0
    print("\nExtracting landmarks for special classes...")

    for label in tqdm(special_classes):
        folder = os.path.join(dataset_root, label)
        if not os.path.exists(folder):
            print(f"Folder not found: {label}")
            continue

        img_count = 0
        detected = 0
        for img_file in os.listdir(folder):
            if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            img_count += 1

            img_path = os.path.join(folder, img_file)
            image = cv2.imread(img_path)
            if image is None:
                continue

            rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

            detection_result = landmarker.detect(mp_image)

            if detection_result.hand_landmarks:
                lm = detection_result.hand_landmarks[0]
                row = []
                for landmark in lm:
                    row.extend([landmark.x, landmark.y, landmark.z])
                row.append(label)  # label = "SPACE", "DELETE", etc.
                writer.writerow(row)
                total_detected_special += 1
                detected += 1

        if img_count > 0:
            print(f"  {label}: {img_count} images → {detected} hands detected")

print(f"Special extraction finished! Rows: {total_detected_special}")


Extracting landmarks for special classes...


 50%|█████     | 1/2 [01:15<01:15, 75.68s/it]

  space: 3000 images → 1375 hands detected


100%|██████████| 2/2 [02:33<00:00, 76.58s/it]

  del: 3000 images → 1663 hands detected
Special extraction finished! Rows: 3038


In [ ]:
output_csv2=os.path.join(local_extract_path, "special_landmarks.csv")

!cp "{output_csv2}" "/content/drive/MyDrive/Colab_Notebooks/EDGE/"

In [ ]:
import pandas as pd

# Load existing A-Z CSV
df_az = pd.read_csv("/content/archive/asl_landmarks.csv")  # your current one

# Load special CSV
df_special = pd.read_csv("/content/archive/special_landmarks.csv")

# Combine (append rows)
df_combined = pd.concat([df_az, df_special], ignore_index=True)

# Optional: shuffle for better training
df_combined = df_combined.sample(frac=1).reset_index(drop=True)

# Save combined
df_combined.to_csv("/content/archive/combined_asl_landmarks.csv", index=False)

print(f"Combined CSV created! Total rows: {len(df_combined)}")
print("Class distribution:\n", df_combined['label'].value_counts())

Combined CSV created! Total rows: 62035
Class distribution:
 label
F        2827
K        2652
Y        2559
V        2519
J        2516
R        2510
S        2500
L        2488
U        2488
W        2423
D        2396
G        2361
H        2339
I        2330
T        2324
Z        2307
E        2270
O        2219
B        2172
A        2143
X        2122
Q        2055
P        2000
C        1877
del      1663
M        1434
space    1375
N        1166
Name: count, dtype: int64


In [ ]:
!cp "/content/archive/combined_asl_landmarks.csv" "/content/drive/MyDrive/Colab_Notebooks/EDGE/"